# Circ1PAT1R Fisher matrix without the LISA response

Computes the Fisher matrix over all parameters of FEW's post-adiabatic model `Circ1PAT1R` (FEW >= 2.2), including the secondary spin `chi2`. SEF uses the sky-averaged LISA sensitivity curve on $h_+$ and $h_\times$.

In [1]:
import numpy as np

import few
from few.waveform import Circ1PAT1R

from stableemrifisher.fisher import StableEMRIFisher
from stableemrifisher.utils.analysis import check_fisher

use_gpu = False  # set to True to run on an NVIDIA GPU

if not use_gpu:
    # restrict FEW to its CPU backend
    cfg_set = few.get_config_setter(reset=True)
    cfg_set.enable_backends("cpu")

## Set up StableEMRIFisher

`evolve_primary=False` holds the primary's mass and spin fixed.

In [2]:
T = 0.1  # observation time, years
dt = 10.0  # sampling interval, seconds

sef = StableEMRIFisher(
    waveform_class=Circ1PAT1R,
    waveform_class_kwargs=dict(
        inspiral_kwargs=dict(
            evolve_primary=False,  # hold the primary's mass and spin fixed
            err=1e-11,  # trajectory integrator tolerance
        ),
    ),
    waveform_kwargs=dict(mode_selection_threshold=1e-5),
    T=T,
    dt=dt,
    deriv_type="stable",
    use_gpu=use_gpu,
)

No noise model or response wrapper provided.
Defaulting to the sky-averaged sensitivity curve


## Compute the full Fisher matrix

The secondary spin `chi2` is passed through `add_param_args`.

In [3]:
wave_params = dict(
    m1=1e6,  # primary mass, solar masses
    m2=1e3,  # secondary mass, solar masses
    a=0.5,  # primary spin
    p0=16.0,  # initial orbital separation
    e0=0.0,  # not used by Circ1PAT1R
    xI0=1.0,  # not used by Circ1PAT1R
    dist=0.1,  # luminosity distance, Gpc
    qS=np.pi / 3,  # sky location, polar angle
    phiS=np.pi / 4,  # sky location, azimuthal angle
    qK=np.pi / 6,  # spin orientation, polar angle
    phiK=np.pi / 8,  # spin orientation, azimuthal angle
    Phi_phi0=1.0,  # initial azimuthal phase
    Phi_theta0=0.0,  # not used by Circ1PAT1R
    Phi_r0=0.0,  # not used by Circ1PAT1R
)
add_param_args = dict(chi2=0.9)  # secondary spin

param_names = ["m1", "m2", "a", "p0", "dist", "qS", "phiS", "qK", "phiK", "Phi_phi0", "chi2"]

Fisher = sef(
    wave_params,
    add_param_args=add_param_args,
    param_names=param_names,
    der_order=4,
    Ndelta=8,
    stats_for_nerds=True,
)

Waveform Generated. SNR: 2785.364506003604
The step chosen for m1 is the largest trial step (1.931e+01): the most stable step may lie outside the grid. Extend delta_range to larger steps.
  m1         delta = 1.931e+01   relerr = 8.1e-12   (8 steps)             
The step chosen for m2 is the largest trial step (1.931e-02): the most stable step may lie outside the grid. Extend delta_range to larger steps.
  m2         delta = 1.931e-02   relerr = 3.9e-10   (8 steps)              
The step chosen for a is the largest trial step (9.653e-06): the most stable step may lie outside the grid. Extend delta_range to larger steps.
  a          delta = 9.653e-06   relerr = 1.3e-09   (8 steps)             
  p0         delta = 5.964e-05   relerr = 3.1e-11   (8 steps)              
  dist       analytic derivative                                           
  qS         delta = 1.455e-04   relerr = 3.4e-13   (8 steps)              
  phiS       delta = 1.091e-04   relerr = 2.4e-13   (8 steps)        

## Degenerate parameters

Without the response, two combinations of the angles `qS`, `phiS`, `qK`, `phiK` have no effect on the waveform. SEF reports them, and any other near-degenerate combination, in `sef.fisher_check`.

In [4]:
check = sef.fisher_check
print("positive definite:", check["positive_definite"])
for params, rel_eigenvalue in check["degenerate"]:
    print(f"unconstrained combination of {params}, relative eigenvalue {rel_eigenvalue:.1e}")

positive definite: False
unconstrained combination of ['qS', 'phiS', 'qK'], relative eigenvalue -2.2e-17
unconstrained combination of ['qS', 'phiS', 'qK', 'phiK'], relative eigenvalue 3.1e-17
unconstrained combination of ['m1', 'p0'], relative eigenvalue 2.2e-14


## Uncertainties with the angles known

Removing the rows and columns of the angles gives an invertible Fisher matrix.

In [5]:
angles = ["qS", "phiS", "qK", "phiK"]
kept = [name for name in param_names if name not in angles]
index = [param_names.index(name) for name in kept]

Fisher_kept = Fisher[np.ix_(index, index)]
print("positive definite:", check_fisher(Fisher_kept, kept)["positive_definite"])

sigma = np.sqrt(np.diag(np.linalg.inv(Fisher_kept)))
for name, s in zip(kept, sigma):
    print(f"{name:9s} = {sef.wave_params[name]:.6g} +/- {s:.3g}")

positive definite: True
m1        = 1e+06 +/- 3.28e+03
m2        = 1000 +/- 2.21
a         = 0.5 +/- 0.0256
p0        = 16 +/- 0.0349
dist      = 0.1 +/- 0.000111
Phi_phi0  = 1 +/- 0.000823
chi2      = 0.9 +/- 49.2
